# Abgleich der beiden Datenquellen

Dieses Notebook lädt die Excel-Datei `data/Bauten_ab_1970.xlsx` und die bereinigte KGWR-CSV aus `output/kgwr/` und sucht nach übereinstimmenden Gebäuden.

## 1) Dateien laden


In [1]:
from pathlib import Path
import re
from typing import Optional, Tuple

import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == 'notebook' else cwd
excel_path = project_root / 'data' / 'Bauten_ab_1970.xlsx'
kgwr_path = project_root / 'output' / 'kgwr' / 'kgwr_gebaeude_luzern_baujahr_1970_2000.csv'

print('Excel:', excel_path)
print('KGWR :', kgwr_path)

if excel_path.stat().st_size == 0:
    raise ValueError('Die Excel-Datei ist leer (0 Bytes). Bitte zuerst die Quelldatei bereitstellen.')

excel_df = pd.read_excel(excel_path)
kgwr_df = pd.read_csv(kgwr_path)

print('Excel-Form:', excel_df.shape)
print('KGWR-Form :', kgwr_df.shape)


Excel: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/data/Bauten_ab_1970.xlsx
KGWR : /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/kgwr/kgwr_gebaeude_luzern_baujahr_1970_2000.csv
Excel-Form: (171, 8)
KGWR-Form : (2406, 77)


## 2) Daten normalisieren


In [2]:
def extract_gemeinde(adresse: object) -> str:
    text = str(adresse).strip()
    if not text or text.lower() == 'nan':
        return ''
    if ',' in text:
        return text.split(',', 1)[0].strip()
    return text.split()[0].strip()

def parse_adresse(adresse: object, gemeinde: object) -> Tuple[Optional[str], Optional[str]]:
    text = str(adresse).strip()
    if not text or text.lower() == 'nan':
        return None, None

    first = text.split('/', 1)[0].strip()
    first = first.split('\t', 1)[0].strip()

    if isinstance(gemeinde, str) and gemeinde.strip():
        prefix = gemeinde.strip() + ' '
        if first.lower().startswith(prefix.lower()):
            first = first[len(prefix):].strip()

    primary = first.split(',', 1)[0].strip()
    match = re.match(r'^(.*?)(?:\s+)(\d[\dA-Za-z\-\/\.]*)$', primary)
    if match:
        raw_number = match.group(2).strip().split(',', 1)[0].strip()
        house_number = re.match(r'^(\d+[A-Za-z]?)(?:\s*[-/].*)?$', raw_number)
        return match.group(1).strip(), house_number.group(1) if house_number else raw_number

    return primary, None

excel_norm = excel_df.copy()
excel_norm['Gemeinde'] = excel_norm['Gemeinde'].fillna('')
mask = excel_norm['Gemeinde'].eq('Agglomeration') & excel_norm['Adresse'].notna()
excel_norm.loc[mask, 'Gemeinde'] = excel_norm.loc[mask, 'Adresse'].apply(extract_gemeinde)

parsed = excel_norm.apply(lambda row: parse_adresse(row['Adresse'], row['Gemeinde']), axis=1)
excel_norm['Gemeinde_norm'] = excel_norm['Gemeinde'].astype(str).str.strip().str.casefold()
excel_norm['STRNAMK1_HPT_norm'] = parsed.map(lambda x: x[0]).fillna('').astype(str).str.strip().str.casefold()
excel_norm['DEINR_norm'] = parsed.map(lambda x: x[1]).fillna('').astype(str).str.strip()

kgwr_norm = kgwr_df.copy()
kgwr_norm['Gemeinde_norm'] = kgwr_norm['BFS_GEMEINDE'].fillna('').astype(str).str.strip().str.casefold()
kgwr_norm['STRNAMK1_HPT_norm'] = kgwr_norm['STRNAMK1_HPT'].fillna('').astype(str).str.strip().str.casefold()
kgwr_norm['DEINR_norm'] = kgwr_norm['DEINR'].fillna('').astype(str).str.strip()
kgwr_norm['DEINR_norm'] = kgwr_norm['DEINR_norm'].str.extract(r'^(\d+[A-Za-z]?)', expand=False).fillna(kgwr_norm['DEINR_norm'])

print('Excel-Normalisierung fertig:', excel_norm.shape)
print('KGWR-Normalisierung fertig :', kgwr_norm.shape)


Excel-Normalisierung fertig: (171, 11)
KGWR-Normalisierung fertig : (2406, 80)


## 3) Übereinstimmungen finden


In [3]:
key_cols = ['Gemeinde_norm', 'STRNAMK1_HPT_norm', 'DEINR_norm']

matches = excel_norm.merge(
    kgwr_norm,
    on=key_cols,
    how='inner',
    suffixes=('_excel', '_kgwr')
)

coverage = excel_norm.merge(
    kgwr_norm[key_cols].drop_duplicates(),
    on=key_cols,
    how='left',
    indicator=True
)

print('Excel-Zeilen:', len(excel_norm))
print('KGWR-Zeilen :', len(kgwr_norm))
print('Matches     :', len(matches))
print('Nur Excel   :', int((coverage['_merge'] == 'left_only').sum()))

display(matches[[
    'Gemeinde', 'Adresse', 'Gemeinde_norm', 'STRNAMK1_HPT_norm', 'DEINR_norm',
    'BFS_GEMEINDE', 'STRNAMK1_HPT', 'DEINR'
]].head(25))


Excel-Zeilen: 171
KGWR-Zeilen : 2406
Matches     : 113
Nur Excel   : 97


,Gemeinde,Adresse,Gemeinde_norm,STRNAMK1_HPT_norm,DEINR_norm,BFS_GEMEINDE,STRNAMK1_HPT,DEINR
0,Luzern,Alpenquai 12-14,luzern,alpenquai,12,Luzern,Alpenquai,12.0
1,Luzern,Alpenquai 20-22,luzern,alpenquai,20,Luzern,Alpenquai,20.0
2,Luzern,Alpenquai 28-30,luzern,alpenquai,28,Luzern,Alpenquai,28.0
3,Luzern,Alpenquai 28-30,luzern,alpenquai,28,Luzern,Alpenquai,28.0
4,Luzern,Alpenquai 34 / Landenbergstrasse 14/16,luzern,alpenquai,34,Luzern,Alpenquai,34.0
5,Luzern,Alpenquai 34 / Landenbergstrasse 14/16,luzern,alpenquai,34,Luzern,Alpenquai,34.0
6,Luzern,Alpenquai 34 / Landenbergstrasse 14/16,luzern,alpenquai,34,Luzern,Alpenquai,34.0
7,Luzern,Alpenquai 34 / Landenbergstrasse 14/16,luzern,alpenquai,34,Luzern,Alpenquai,34.0
8,Luzern,Bahnhofplatz 1,luzern,bahnhofplatz,1,Luzern,Bahnhofplatz,1.0
9,Luzern,Bahnhofplatz 1,luzern,bahnhofplatz,1,Luzern,Bahnhofplatz,1.0


## 4) Nicht übereinstimmende Einträge anzeigen


In [4]:
excel_only = coverage.loc[coverage['_merge'] == 'left_only'].copy()
excel_only = excel_only[['Gemeinde', 'Adresse', 'STRNAMK1_HPT_norm', 'DEINR_norm']].head(25)

kgwr_only = kgwr_norm.merge(
    excel_norm[key_cols].drop_duplicates(),
    on=key_cols,
    how='left',
    indicator=True
)
kgwr_only = kgwr_only.loc[kgwr_only['_merge'] == 'left_only', ['BFS_GEMEINDE', 'STRNAMK1_HPT', 'DEINR']].head(25)

print('Nur Excel (erste 25):')
display(excel_only)
print('Nur KGWR (erste 25):')
display(kgwr_only)


Nur Excel (erste 25):


,Gemeinde,Adresse,STRNAMK1_HPT_norm,DEINR_norm
4,Luzern,Alpenquai 36-40 / Landenbergstrasse 8-12,alpenquai,36
5,Luzern,Alpenquai 42,alpenquai,42
6,Luzern,Alpenstrasse 12,alpenstrasse,12
13,Luzern,"Blattenmoos 8, 8a, 8b",blattenmoos,8
15,Luzern,Brünigstrasse 16-18,brünigstrasse,16
16,Luzern,Bundesplatz 15,bundesplatz,15
18,Luzern,Bürgenstrasse 12/14 / Rösslimattstrasse 37,bürgenstrasse,12
19,Luzern,Burgerstrasse 20,burgerstrasse,20
22,Luzern,Eggen 5,eggen,5
23,Luzern,Europaplatz,europaplatz,


Nur KGWR (erste 25):


,BFS_GEMEINDE,STRNAMK1_HPT,DEINR
0,Luzern,Bennenegg,22.0
1,Luzern,Bennenegg,24.0
2,Luzern,Bennenegg,26.0
3,Luzern,Bennenegg,28.0
4,Luzern,Bennenegg,30.0
5,Luzern,Bennenegg,32.0
6,Luzern,Löwengrube,12.0
7,Luzern,Löwengrube,10.0
8,Luzern,Löwengrube,4.0
9,Luzern,Bennenegg,14.0
